In [ ]:
# Resolve repository imports and data paths when opened from notebooks/.
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "sae").is_dir():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Run this notebook from the KronSAE repository.")
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
import torch

def mand_op(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Implements the AND operation in logit space."""
    # see https://arxiv.org/pdf/2110.11940 for analytic derivation
    logical_acts = torch.sqrt(torch.relu(u) * torch.relu(v))
    return logical_acts

def and_op(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Implements the AND operation in logit space."""
    # see https://arxiv.org/pdf/2110.11940 for analytic derivation
    cond = (u < 0) & (v < 0)
    logical_acts = torch.where(cond, u + v, torch.minimum(u, v)).type_as(u)
    return logical_acts


In [10]:
num_range = 4
n = 100
x = (torch.arange(n * num_range + 1) - (n * num_range) / 2) / n
y = (torch.arange(n * num_range + 1) - (n * num_range) / 2) / n


In [ ]:
import matplotlib.pyplot as plt
import torch
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

fig, axs = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax in axs:
    #ax.minorticks_on()
    ax.tick_params(axis='y', which='major',  left=True, right=False,  labelsize=14)
    ax.tick_params(axis='y', which='minor',  left=True, right=False)
    ax.tick_params(axis='x', which='major',  bottom=True, top=False, labelsize=14)
    ax.tick_params(axis='x', which='minor',  bottom=True, top=False)

    # ← NEW: force integer steps and integer labels
    ax.xaxis.set_major_locator(MultipleLocator(0.5))
    ax.yaxis.set_major_locator(MultipleLocator(0.5))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))

X, Y = torch.meshgrid(x, y)
Zmand = mand_op(X, Y)
Zand  = and_op(X, Y)

# left panel: mAND
cs1 = axs[0].contourf(
    X.numpy(), Y.numpy(), Zmand.numpy(),
    levels=50, cmap='vlag'
)
axs[0].set_title(r"$mAND$", fontsize = 16)
axs[0].set_xlabel(r"$\mathbf{p}$", fontsize = 14)      # bold p on x
axs[0].set_ylabel(r"$\mathbf{q}$", rotation=0, fontsize = 14)      # bold q on y
axs[0].tick_params(axis='y', which='major', labelsize=14)
axs[0].tick_params(axis='x', which='major', labelsize=14)
#axs[0].minorticks_on()
# right panel: AND
cs2 = axs[1].contourf(
    X.numpy(), Y.numpy(), Zand.numpy(),
    levels=100, cmap='vlag'
)

axs[1].set_title("$AND_{AIL}$", fontsize=16)
axs[1].set_xlabel(r"$\mathbf{p}$", fontsize=14)      # bold p on x
axs[1].tick_params(axis='y', which='major', labelsize=14)
axs[1].tick_params(axis='x', which='major', labelsize=14)
axs[1].set_xlim(-2, 2)
axs[1].set_ylim(-2, 2)
#axs[1].minorticks_on()
# no need to set_ylabel here because sharey=True

# single shared colorbar
# we need to pick one of the mappable objects (cs1 or cs2) and tell
# colorbar which axes to span:
cbar_ax = fig.add_axes([0.95, 0.11, 0.02, 0.77])  # [left, bottom, width, height]

cbar = fig.colorbar(
    cs1, cax=cbar_ax,                       # any of the two contour sets
    ax=axs,                    # list of all subplots
    orientation="vertical",    # or 'horizontal'
    fraction=0.046,            # shrink colorbar to fit
    pad=0.04                   # spacing from plots
)
cbar.set_label("Activation value", fontsize=14)  # optional

#plt.tight_layout()
fig.savefig("mand.pdf", bbox_inches="tight")
plt.show()
